# 03c - RM-c: frozen encoder + Retrieval-Augmented Classification

RM-c tidak melatih apa pun. Ia memakai ulang dua artefak milik RM-b, yaitu
embedding beku dan head yang sudah terlatih, lalu menambahkan cabang retrieval.

Alur per sampel:

1. `p_bert` = softmax(head(embedding))
2. Cari k tetangga terdekat di indeks FAISS yang dibangun HANYA dari split train,
   lalu ubah label tetangga menjadi distribusi `p_retr`
3. `p_final = (1 - alpha) * p_bert + alpha * p_retr`, prediksi = argmax

Fusi dilakukan pada level probabilitas dan softmax hanya diterapkan sekali, di
cabang BERT sebelum fusi. Karena kedua masukan sudah berupa distribusi dan bobot
fusinya berjumlah satu, hasilnya sudah menjadi distribusi sah; softmax kedua
hanya akan meratakan selisih dan bisa mengubah argmax pada kasus nyaris seri.

Prasyarat: `03b_rmb_frozen.ipynb` sudah dijalankan (butuh `rmb_best.pt`).

In [1]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-14 16:50:15,881 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device        : cuda
encoder       : indobenchmark/indobert-base-p2
keluaran      : /workspace/indobert-with-rac/outputs/baseline
train/val/test: [6588, 1402, 1405]


## 1. Head RM-b yang diwarisi

In [2]:
head, head_config = runner._load_best_head()
print("konfigurasi head RM-b terbaik:", head_config)
print("indeks FAISS akan dibangun dari", len(runner.features.labels["train"]), "vektor train")

konfigurasi head RM-b terbaik: {'head_arch': 'mlp', 'hidden_dim': 256, 'epochs': 5, 'lr': 0.0002, 'dropout': 0.1, 'weight_decay': 0.01, 'batch': 32, 'seed': 42}
2026-09-14 16:50:17,228 | INFO     | src.services.features | Fitur beku dimuat dari cache /workspace/indobert-with-rac/outputs/baseline/features/indobenchmark__indobert-base-p2
indeks FAISS akan dibangun dari 6588 vektor train


Indeks dibangun eksklusif dari split train. Kalau val atau test ikut masuk,
retrieval akan menemukan sampel uji di dalam indeksnya sendiri dan hasilnya
kehilangan makna.

## 2. Konfigurasi

In [3]:
from src.models.schemas import RMCConfig

config = RMCConfig()
print(config.model_dump())

{'alpha': 0.3, 'k': 5, 'weighting': 'similarity'}


## 3. Jalankan

In [4]:
row = runner.run(
    "rmc",
    config.model_dump(),
    note="baseline RAC alpha=0,3 k=5 mengikuti Yu dkk. 2023",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro    : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi     : {row['val_f1_judi']:.4f}")
print(f"  vektor di indeks: {row['index_vectors']:,}")
print(f"  waktu evaluasi  : {row['eval_time_s']:.2f} s")
print(f"  trainable params: {row['trainable_params']}")
print(f"  waktu latih     : {row['train_time_s']}")

2026-09-14 16:50:17,237 | INFO     | src.services.campaign | [rmc] RUN #1 {'alpha': 0.3, 'k': 5, 'weighting': 'similarity'}
2026-09-14 16:50:17,358 | INFO     | src.services.rac | Indeks FAISS dibangun: 6588 vektor berdimensi 768
2026-09-14 16:50:17,579 | INFO     | src.services.run_log | Juara baru untuk rmc: val F1-macro 0.9540 (sebelumnya -1.0000)
2026-09-14 16:50:17,581 | INFO     | src.services.campaign | [rmc] RUN #1 val F1-macro 0.9540
run #1
  val F1-macro    : 0.9540
  val F1 judi     : 0.9251
  vektor di indeks: 6,588
  waktu evaluasi  : 0.11 s
  trainable params: 0
  waktu latih     : 0.0


## 4. Pengaruh alpha

In [5]:
import pandas as pd

sweep = runner.run_batch(
    "rmc",
    [
        {"config": {"alpha": alpha, "k": 5},
         "note": f"sweep alpha={alpha} pada k=5"}
        for alpha in (0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0)
    ],
    batch_id="rmc_sweep_alpha",
)

sweep[["run_id", "alpha", "k", "val_f1_macro", "val_f1_judi"]]

2026-09-14 16:50:17,594 | INFO     | src.services.campaign | Batch rmc_sweep_alpha: 1 dari 7 konfigurasi sudah ada di riwayat, dilewati
2026-09-14 16:50:17,594 | INFO     | src.services.campaign | Batch rmc_sweep_alpha: 6 konfigurasi akan dijalankan
2026-09-14 16:50:17,595 | INFO     | src.services.campaign | Batch rmc_sweep_alpha: 1/6 (perkiraan sisa 0.0 menit)
2026-09-14 16:50:17,597 | INFO     | src.services.run_log | Riwayat runs_rmc.csv dimuat: 1 run
2026-09-14 16:50:17,597 | INFO     | src.services.campaign | [rmc] RUN #2 {'alpha': 0.0, 'k': 5, 'weighting': 'similarity'}
2026-09-14 16:50:17,623 | INFO     | src.services.rac | Indeks FAISS dibangun: 6588 vektor berdimensi 768
2026-09-14 16:50:17,679 | INFO     | src.services.campaign | [rmc] RUN #2 val F1-macro 0.9400
2026-09-14 16:50:17,679 | INFO     | src.services.campaign | Batch rmc_sweep_alpha: 2/6 (perkiraan sisa 0.0 menit)
2026-09-14 16:50:17,682 | INFO     | src.services.run_log | Riwayat runs_rmc.csv dimuat: 2 run
2026-0

,run_id,alpha,k,val_f1_macro,val_f1_judi
0,2,0.0,5,0.940029,0.902985
1,3,0.1,5,0.946489,0.913208
2,4,0.2,5,0.948667,0.916667
3,5,0.5,5,0.962449,0.938370
4,6,0.7,5,0.950856,0.919028
5,7,1.0,5,0.941413,0.903093


`alpha=0` identik dengan RM-b murni dan `alpha=1` membuang cabang BERT
sepenuhnya, sehingga kedua ujung itu berfungsi sebagai pemeriksaan kewarasan:
kolom pertama harus sama persis dengan F1 RM-b.

In [6]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep["alpha"], sweep["val_f1_macro"], marker="o", label="F1-macro")
ax.plot(sweep["alpha"], sweep["val_f1_judi"], marker="s", label="F1 judi")
ax.set_xlabel("alpha (bobot cabang retrieval)")
ax.set_ylabel("F1 (validation)")
ax.set_title("Pengaruh bobot fusi RAC")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Ringkasan

RM-c menambah nol trainable parameter dan nol waktu latih di atas RM-b; seluruh
biaya tambahannya ada di inferensi, yaitu pembangunan indeks sekali dan
penelusuran k tetangga per prediksi. Biaya itu diukur terpisah di
`06_analysis_export.ipynb`.

Lanjut ke `04_tuning_campaign.ipynb`.